# QAOA Replication: MaxCut on a Triangle Graph
Based on Chapter 9 of *Quantum Algorithms from First Principles* and Farhi et al. (2014).

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Step 1: Define the Problem
n_qubits = 3
edges = [(0, 1), (1, 2), (0, 2)]
print(f"Graph initialized: {n_qubits} vertices, edges: {edges}")

Graph initialized: 3 vertices, edges: [(0, 1), (1, 2), (0, 2)]


### Step 2: Build the QAOA Circuit
We implement the cost layer with a CNOT-$R_z$-CNOT sandwich and the mixer layer with $R_x(-2\beta)$.

In [2]:
def build_qaoa_circuit(n, edges, gammas, betas):
    p = len(gammas)
    qc = QuantumCircuit(n)
    
    # Initial state
    qc.h(range(n))
    
    for layer in range(p):
        gamma = gammas[layer]
        beta = betas[layer]
        
        # Cost Layer
        for i, j in edges:
            qc.cx(i, j)
            qc.rz(gamma, j)
            qc.cx(i, j)
            
        # Mixer Layer (using Rx(-2*beta) as per Chapter 9 conventions)
        for i in range(n):
            qc.rx(-2.0 * beta, i)
            
    return qc

### Step 3: Evaluate p=1 with Optimal Angles
Optimal analytical angles: $\gamma^* = \arctan(1/\sqrt{2})$, $\beta^* = \gamma^*/2$.

In [3]:
gamma_opt = np.arctan(1.0 / np.sqrt(2.0))
beta_opt = gamma_opt / 2.0

qc_p1 = build_qaoa_circuit(n_qubits, edges, [gamma_opt], [beta_opt])
state = Statevector.from_instruction(qc_p1)
probs = state.probabilities_dict()

print(f"Optimal Gamma: {gamma_opt:.4f}, Optimal Beta: {beta_opt:.4f}")
print("Measurement Probabilities:")
for bitstring, p_val in sorted(probs.items(), key=lambda x: x[1], reverse=True):
    print(f"  |{bitstring}⟩: {p_val * 100:.2f}%")

Optimal Gamma: 0.6155, Optimal Beta: 0.3077
Measurement Probabilities:
  |011⟩: 16.67%
  |100⟩: 16.67%
  |001⟩: 16.67%
  |010⟩: 16.67%
  |101⟩: 16.67%
  |110⟩: 16.67%
  |000⟩: 0.00%
  |111⟩: 0.00%


### Step 4: Fixed-Schedule Adiabatic Multi-Layer Simulation
Simulating with a linear adiabatic schedule: $\gamma_j = (j/p)\Delta t$, $\beta_j = (1 - j/p)\Delta t$.

In [5]:
T_total = 4.0
depths = [1, 2, 4, 8, 16, 32, 64]

print("--- Fixed-Schedule Depth Scan (T = 4.0) ---")
for p in depths:
    # Fix 1: Calculate the correct time slice dt = T / p
    dt = T_total / p

    # Fix 2: Apply dt to the schedule, not T_total
    gammas_sched = [(j / p) * dt for j in range(1, p + 1)]
    betas_sched = [(1.0 - j / p) * dt for j in range(1, p + 1)]

    qc_sched = build_qaoa_circuit(n_qubits, edges, gammas_sched, betas_sched)
    sv = Statevector.from_instruction(qc_sched)

    expected_cut = 0.0
    # Fix 3: Use probabilities_dict() to bypass the deprecated data_dict()
    for bs, prob in sv.probabilities_dict().items():
        # Fix 4: Reverse the bitstring to match the textbook's big-endian math convention
        bits = [int(b) for b in bs[::-1]]
        cut_val = sum(1 for i, j in edges if bits[i] != bits[j])
        expected_cut += prob * cut_val

    approx_ratio = expected_cut / 2.0
    print(
        f"Depth p = {p:2d} | Expected Cut Score: {expected_cut:.4f} | Approx Ratio: {approx_ratio:.4f}"
    )

--- Fixed-Schedule Depth Scan (T = 4.0) ---
Depth p =  1 | Expected Cut Score: 1.5000 | Approx Ratio: 0.7500
Depth p =  2 | Expected Cut Score: 0.1057 | Approx Ratio: 0.0529
Depth p =  4 | Expected Cut Score: 1.9782 | Approx Ratio: 0.9891
Depth p =  8 | Expected Cut Score: 1.9803 | Approx Ratio: 0.9902
Depth p = 16 | Expected Cut Score: 1.9772 | Approx Ratio: 0.9886
Depth p = 32 | Expected Cut Score: 1.9745 | Approx Ratio: 0.9873
Depth p = 64 | Expected Cut Score: 1.9728 | Approx Ratio: 0.9864
